In [ ]:
import pandas as pd
import pathlib

base = pathlib.Path("input")
out = pathlib.Path("output")
out.mkdir(exist_ok=True)

use_cases = ["bee-use-case", "cloud-use-case", "marine-use-case", "termite-use-case"]
display_names = {
    "bee-use-case": "Bee",
    "cloud-use-case": "Cloud",
    "marine-use-case": "Marine",
    "termite-use-case": "Termite",
}

def extract_rank(classification, rank):
    """Extract a GTDB rank from a classification string."""
    prefix_map = {"phylum": "p__", "genus": "g__", "species": "s__"}
    prefix = prefix_map.get(rank)
    if pd.isna(classification) or not prefix:
        return None
    for part in str(classification).split(";"):
        part = part.strip()
        if part.startswith(prefix):
            name = part[len(prefix):].strip()
            return name if name else "unclassified"
    return "unclassified"

def top_n_str(series, n=10):
    """Return '; ' joined string of top-n value counts."""
    counts = series.value_counts()
    return "; ".join(f"{s} ({c})" for s, c in counts.head(n).items())

summary_rows = []
per_ucase_tables = {}

for uc in use_cases:
    dat = base / uc / "data"
    uc_name = display_names[uc]

    # --- dRep ---
    drep = pd.read_csv(dat / "drep.csv")
    drep["genome"] = drep["genome"].str.replace(r"\.fasta$", "", regex=True)
    total_mags = len(drep)
    n_clusters = drep["secondary_cluster"].nunique()
    cluster_sizes = drep.groupby("secondary_cluster").size()
    multi_member = (cluster_sizes >= 2).sum()

    # --- CheckM2 & GTDB (cluster reps) ---
    checkm = pd.read_csv(dat / "checkm2.tsv", sep="\t")
    checkm["Name"] = checkm["Name"].str.replace(r"\.fasta$", "", regex=True)
    gtdb = pd.read_csv(dat / "gtdb.tsv", sep="\t")
    gtdb["user_genome"] = gtdb["user_genome"].str.replace(r"\.fasta$", "", regex=True)
    merged = pd.merge(checkm, gtdb, left_on="Name", right_on="user_genome", how="left")
    merged["phylum"] = merged["classification"].apply(lambda x: extract_rank(x, "phylum"))
    merged["genus"] = merged["classification"].apply(lambda x: extract_rank(x, "genus"))
    merged["species"] = merged["classification"].apply(lambda x: extract_rank(x, "species"))
    merged.loc[merged["species"].isna(), "species"] = "no GTDB hit"

    # --- QUAST genome size ---
    quast = pd.read_csv(dat / "quast.tsv", sep="\t", index_col=0).T
    quast.index = quast.index.str.replace(r"\.fasta$", "", regex=True)
    quast["N50_kb"] = (pd.to_numeric(quast["N50"]) / 1e3).round(1)
    quast["L50"] = pd.to_numeric(quast["L50"])
    merged = pd.merge(merged, quast[["L50", "N50_kb"]], left_on="Name", right_index=True, how="left")

    # --- Quality tiers (MIMAG) ---
    n_near_complete = ((merged["Completeness"] > 95) & (merged["Contamination"] < 5)).sum()
    n_high = ((merged["Completeness"] > 90) & (merged["Contamination"] < 5)).sum()
    n_medium = ((merged["Completeness"] >= 50) & (merged["Contamination"] < 10)).sum()
    n_low = len(merged) - n_medium
    n_medium_only = n_medium - n_high  # medium but not high

    # --- Robust completeness/contamination stats ---
    comp_med = merged["Completeness"].median()
    comp_q1 = merged["Completeness"].quantile(0.25)
    comp_q3 = merged["Completeness"].quantile(0.75)
    cont_med = merged["Contamination"].median()
    cont_q1 = merged["Contamination"].quantile(0.25)
    cont_q3 = merged["Contamination"].quantile(0.75)

    # --- L50 / N50 stats ---
    l50 = merged["L50"].dropna()
    n50 = merged["N50_kb"].dropna()

    # --- Unclassified stats ---
    species_counts = merged["species"].value_counts()
    n_unclassified = species_counts.get("unclassified", 0)
    n_no_gtdb = species_counts.get("no GTDB hit", 0)

    # --- Top 30 clusters by size (with taxonomy) ---
    def tax_label(classification):
        """Return lowest resolved rank with GTDB prefix if not species-level."""
        if pd.isna(classification):
            return "no GTDB hit"
        ranks = [("s__", None), ("g__", "g__"), ("f__", "f__"), ("p__", "p__")]
        for prefix, label in ranks:
            for part in str(classification).split(";"):
                part = part.strip()
                if part.startswith(prefix):
                    name = part[len(prefix):].strip()
                    if name:
                        return name if label is None else f"{label}{name}"
        return "no GTDB hit"

    drep_merged = pd.merge(drep, gtdb[["user_genome", "classification"]],
                           left_on="genome", right_on="user_genome", how="left")
    cluster_tax = drep_merged.groupby("secondary_cluster").agg(
        size=("genome", "size"),
        classification=("classification", "first"),
    ).sort_values("size", ascending=False)
    cluster_tax["label"] = cluster_tax["classification"].apply(tax_label)
    cluster_tax = cluster_tax[cluster_tax["label"] != "no GTDB hit"]
    top30_taxa = "; ".join(
        f"{row['label']} ({row['size']})"
        for _, row in cluster_tax.head(30).iterrows()
    )

    summary_rows.append({
        "Use case": uc_name,
        "Total MAGs": total_mags,
        "Species-level clusters": n_clusters,
        "Clusters >=2": multi_member,
        "Avg completeness": f"{merged['Completeness'].mean():.1f}",
        "Completeness range": f"{merged['Completeness'].min():.1f}–{merged['Completeness'].max():.1f}",
        "Avg contamination": f"{merged['Contamination'].mean():.2f}",
        "Contamination range": f"{merged['Contamination'].min():.2f}–{merged['Contamination'].max():.2f}",
        "Top 30 taxa": top30_taxa,
    })

    # --- Per-use-case all cluster reps ---
    reps = merged.sort_values("Completeness", ascending=False).copy()
    cluster_size_map = cluster_sizes.to_dict()
    reps["Cluster members"] = reps["Name"].map(
        lambda n: cluster_size_map.get(
            drep.loc[drep["genome"] == n, "secondary_cluster"].iloc[0], 0
        ) if n in drep["genome"].values else 0
    )
    def lowest_tax(row):
        for col, label in [("species", None), ("genus", "genus"), ("phylum", "phylum")]:
            val = row[col]
            if pd.notna(val) and val not in ("unclassified", "no GTDB hit"):
                return val if label is None else f"{val} ({label})"
        return row.get("species", "no GTDB hit")
    reps["Taxonomy"] = reps.apply(lowest_tax, axis=1)
    reps["L50"] = reps["L50"].astype(int)
    reps["N50 (kb)"] = reps["N50_kb"]
    reps = reps[["Name", "Taxonomy", "Cluster members", "L50", "N50 (kb)", "Completeness", "Contamination"]]
    reps.columns = ["MAG", "Taxonomy", "Cluster members", "L50", "N50 (kb)", "Completeness", "Contamination"]
    reps["Completeness"] = reps["Completeness"].round(1).astype(str) + "%"
    reps["Contamination"] = reps["Contamination"].round(2).astype(str) + "%"
    per_ucase_tables[uc_name] = reps.reset_index(drop=True)

# --- Assemble & save summary ---
summary = pd.DataFrame(summary_rows)
summary_cols = [
    "Use case", "Total MAGs", "Species-level clusters", "Clusters >=2",
    "Avg completeness", "Completeness range",
    "Avg contamination", "Contamination range",
    "Top 30 taxa",
]
summary = summary[summary_cols]

summary.to_csv(out / "summary.tsv", sep="\t", index=False)

# --- Table: Top 30 taxa + All phyla ---
top30_only = summary[["Use case", "Top 30 taxa"]].copy()

# --- All phyla per use case (total MAGs per phylum) ---
phyla_rows = []
for uc in use_cases:
    dat = base / uc / "data"
    uc_name = display_names[uc]
    drep = pd.read_csv(dat / "drep.csv")
    drep["genome"] = drep["genome"].str.replace(r"\.fasta$", "", regex=True)
    cluster_sizes_all = drep.groupby("secondary_cluster").size().to_dict()
    gtdb = pd.read_csv(dat / "gtdb.tsv", sep="\t")
    gtdb["user_genome"] = gtdb["user_genome"].str.replace(r"\.fasta$", "", regex=True)
    gtdb["phylum"] = gtdb["classification"].apply(lambda x: extract_rank(x, "phylum"))
    gtdb_phyl = gtdb[gtdb["phylum"].notna() & (gtdb["phylum"] != "unclassified")][["user_genome", "phylum"]]
    drep_gtdb = pd.merge(drep, gtdb_phyl, left_on="genome", right_on="user_genome", how="inner")
    cluster_phyla = drep_gtdb.drop_duplicates("secondary_cluster")[["secondary_cluster", "phylum"]]
    cluster_phyla["cluster_size"] = cluster_phyla["secondary_cluster"].map(cluster_sizes_all)
    counts = cluster_phyla.groupby("phylum")["cluster_size"].sum().sort_values(ascending=False)
    all_phyla = "; ".join(f"{p} ({c})" for p, c in counts.items())
    phyla_rows.append({"Use case": uc_name, "All phyla": all_phyla})
phyla_df = pd.DataFrame(phyla_rows)

taxa_phyla = pd.merge(top30_only, phyla_df, on="Use case")
taxa_phyla.to_csv(out / "taxa_phyla.tsv", sep="\t", index=False)
taxa_phyla

In [ ]:
# All cluster reps per use case (sorted by completeness)
for name, tbl in per_ucase_tables.items():
    print(f"\n=== {name} ===")
    print(tbl.to_string(index=False))
    tbl.to_csv(out / f"reps_{name.lower()}.tsv", sep="\t", index=False)

## Discussion: completeness & contamination — metrics and improvement

### Better representation of quality
- **Median + IQR** (shown above) is more robust than mean for skewed distributions.
- **MIMAG quality tiers** (near-complete/high/medium/low) provide actionable thresholds.
- The Cloud use case is dominated by low-quality MAGs (median completeness ~56%). Showing only
  mean/min/max hides the bimodal nature — a few high-quality reps pull the max to 100%.

### How to improve completeness
- **Deeper sequencing** — especially for Cloud (low biomass, high strain diversity).
- **Co-assembly** of related samples boosts contiguity for shared community members.
- **Iterative binning** (MetaBAT2 + CONCOCT + MaxBin → DAS_Tool / MetaWRAP) recovers more
  genomes per sample.
- **Long reads** (Oxford Nanopore / PacBio HiFi) close fragmented bins.
- **Differential coverage** across multiple samples is the single strongest signal for binning.

### How to reduce contamination
- **RefineM / MAGIC / GUNC** identify chimeric bins by GC, coverage, and taxonomic marker
  consistency.
- **Manual curation** via anvi'o or VizBin for high-priority clusters (e.g., novel species).
- **Tighter binning parameters** (e.g., MetaBAT2 `--minCV` for stricter coverage variance).
- The Cloud use-case shows several reps with 10–25% contamination despite low completeness —
  suggesting chimeric bins from diverse strain populations.
- **Single-copy marker gene counts** (CheckM/CheckM2) flag multi-contamination early.